# GPT API Tests

In [ ]:
# Run
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Imports

In [ ]:
# Run
import base64
import requests
import os
import tqdm
import re
import pandas as pd
import json
import numpy as np
import json
from copy import deepcopy
import ast

### Functions

#### Image Encoding

In [ ]:
# Run
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
# Run
def encode_image2(image_path, basewidth=512):
    # Resize the image using the existing resize_image function
    resized_np_array = resize_image(image_path, basewidth)

    # Convert the numpy array back to a PIL Image
    resized_pil_image = Image.fromarray(resized_np_array)

    # Save the PIL image to an in-memory byte stream
    byte_stream = io.BytesIO()
    # Using PNG format for saving to byte stream, which is lossless.
    resized_pil_image.save(byte_stream, format='PNG')
    byte_stream.seek(0) # Rewind the stream to the beginning

    # Encode the image from the byte stream
    return base64.b64encode(byte_stream.read()).decode('utf-8')

#### Payload

In [ ]:
# Run
def get_payload(image_path,prompt_text,detail="auto",encode_function=0):
    if encode_function == 0:
      base64_image = encode_image(image_path)
    else:
      base64_image = encode_image2(image_path)

    payload = {
      "model": "gpt-4o",
        "temperature": 0.3,
      "messages": [
        {
          "role": "user",
          "content": [
            {
              "type": "text",
                "text": prompt_text
            },
            {
              "type": "image_url",
              "image_url": {
                "url": f"data:image/jpeg;base64,{base64_image}",
                  "detail":detail
              }
            }
          ]
        }
      ],
      "max_tokens": 400
    }

    return payload

#### API Key

In [ ]:
# Run
def get_file_contents_apikey(filename):
    """ Given a filename,
        return the contents of that file
    """
    try:
        with open(filename, 'r') as f:
            # It's assumed our file contains a single line,
            # with our API key
            return f.read().strip()
    except FileNotFoundError:
        print("'%s' file not found" % filename)

#### LLM Json Output Extraction

In [ ]:
# Run
def reformat_output(result):
    exception = False
    try:
        r1 = re.search('json',result).span()[1]
    except:
        r1 = re.search('\n{\n',r).span()[0]
    try:
        r2 = re.search('"\n}\n',result).span()[1]
        output = result[r1:r2]
    except:
        try:
            r2 = re.search('"\n}',result).span()[1]
            output = result[r1:r2]
        except:
            output = result[r1:]+'"\n}'
            exception = True

    return output, exception

### LLM Headers and Keys

In [ ]:
# Run
path_to_keyfile = 'API key.txt' # e.g. "Documents/apikey.txt"
api_key = get_file_contents_apikey(path_to_keyfile)

In [ ]:
# Run
headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

### Prompt

In [ ]:
prompt = """
Extract all the data from this card. There are 7 main categories: Registration Number, Species, Locality, Collector, Date, Set Mark, and Number of Eggs.
Sometimes the text on the card is printed and sometimes is handwritten — make sure to return both the printed and handwritten text. In the cases where the OCR fails on the handwritten text, use manual transcription.
Return results as a JSON.
"""

In [ ]:
# Run
prompt = """ These are images of bird specimen labels. Extract the information from the labels. There are 10
categories that should be extracted from the label:
1. Museum Registration Number (e.g., 1931.8.18.XX)
2. Genus
3. Species
4. Subspecies
5. Sex (use M and F)
6. Date (in verbatim format)
7. Locality (e.g., “20 km N.W. of”)
8. Place Name
9. Region (e.g., “N. Madagascar”)
10. Expedition title (Archbold-Vernay Exped.)
Prioritize precision, not efficiency. If a data point is indecipherable or you are unable to come to a
confident conclusion, leave it blank. However, DO NOT leave place name blank UNLESS it is missing
from the label. Some labels misspell historical place names in Madagascar, like Lac Iotry,
Manjakatompo, Tsimanampetsotsa, and Iampasika, on multiple occasions. What I listed are the correct
spellings. Use them. Some labels have information crossed out or edited. Some labels have corrected
the information that was crossed out, and others don’t. If there is no correction on the label, leave the
information that was crossed out blank in the output. Make sure to use “M” for the male symbol and “F”
for the female symbol. In some instances, there will be a question mark or “juv.” after the sex symbol.
Return only a valid JSON object with the following keys, and nothing else:
["RegistrationNumber", "Genus", "Species", "Subspecies", "Sex", "Date", "Locality", "PlaceName",
"Region", "Expedition"] Do not include any explanation, markdown, or formatting. Do not wrap it in triple
backticks. Just return a single clean JSON object."""

### Single Test Example

In [ ]:
path_to_image = '/content/Photos/1931.8.18.1866.JPG'
payload = get_payload(path_to_image,prompt)
response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

In [ ]:
print(response.json())

{'id': 'chatcmpl-DAuZMfFiRbDnYFsFN779SbhEfF2QX', 'object': 'chat.completion', 'created': 1771493104, 'model': 'gpt-4o-2024-08-06', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '{\n    "RegistrationNumber": "1931.8.18.1866",\n    "Genus": "Zoothera",\n    "Species": "grandidieri",\n    "Subspecies": "",\n    "Sex": "M",\n    "Date": "23 July 1929",\n    "Locality": "30 km west",\n    "PlaceName": "Vondrozo",\n    "Region": "S.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}', 'refusal': None, 'annotations': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 1168, 'completion_tokens': 104, 'total_tokens': 1272, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}, 'service_tier': 'default', 'system_fingerprint': 'fp_ad98c18a04'}


### Looped Tests

In [ ]:
path_to_folder = 'Photos'

all_new_responses = {}

errors = []

k = 10 # Looks at first k images. If interested in whole folder then remove [:k] from below.

for image in tqdm.tqdm(os.listdir(path_to_folder)[:k]):
    if image != '.DS_Store':
        try:
            payload = get_payload(path_to_folder+'/'+image,prompt)
            response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
            text = response.json()['choices'][0]['message']['content']
            all_new_responses[image] = text
        except:
            errors.append(image)

100%|██████████| 5/5 [00:23<00:00,  4.62s/it]


#### Format Outputs -- Version 1

In [ ]:
outputs_formatted = []
formatting_errors = []

for file in all_new_responses.keys():
    try:
        result,_ = reformat_output(all_new_responses[file])
        outputs_formatted.append(json.loads(result))
    except:
        formatting_errors.append(file)


#### Save CSV

In [ ]:
# Save into dataframe
outputs_df = pd.DataFrame(outputs_formatted)
outputs_df.insert(0, 'Filename', list(all_new_responses.keys()))

# Save as CSV
outputs_df.to_csv('/content/drive/My Drive/Madagascar birds/llm_test_results.csv',index=False)

ValueError: Length of values (7) does not match length of index (6)

In [ ]:
# Run
folder_path='/content/drive/My Drive/Madagascar birds/Final cropped images'

In [ ]:
# Run
folders=os.listdir(folder_path)

In [ ]:
# Run
from PIL import Image
def resize_image(image_path,basewidth=1024):
    image = PIL.Image.open(image_path)
    if image.size[0] > basewidth:
        wpercent = (basewidth / float(image.size[0]))
        hsize = int((float(image.size[1]) * float(wpercent)))
        img = image.resize((basewidth, hsize), PIL.Image.LANCZOS)
        I = np.asarray(img)
    else:
        I = np.asarray(image)
    return I

In [ ]:
# Run
import base64
import numpy as np
import io

In [ ]:
# Run
import PIL

## Size test


In [ ]:
sample_path ='Photos'
original_image_outputs = []
resized_image_outputs = []

for file in os.listdir(sample_path):
  payload = get_payload(sample_path+'/'+file,prompt,encode_function=0)
  response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
  text1 = response.json()['choices'][0]['message']['content']
  original_image_outputs.append(text1)
  payload = get_payload(sample_path+'/'+file,prompt,encode_function=1)
  response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
  text2 = response.json()['choices'][0]['message']['content']
  resized_image_outputs.append(text2)

In [ ]:
original_image_outputs

['{\n    "RegistrationNumber": "1931.8.18.353",\n    "Genus": "Tyto",\n    "Species": "alba",\n    "Subspecies": "affinis (hypermetra)",\n    "Sex": "M",\n    "Date": "18 May 1930",\n    "Locality": "",\n    "PlaceName": "Maroantsetra",\n    "Region": "N.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n    "RegistrationNumber": "1931.8.18.1866",\n    "Genus": "Zoonavena",\n    "Species": "grandidieri",\n    "Subspecies": "",\n    "Sex": "M",\n    "Date": "23 July 1929",\n    "Locality": "30 km west",\n    "PlaceName": "Vondrozo",\n    "Region": "S.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n    "RegistrationNumber": "1931.8.18.833",\n    "Genus": "Ixeron",\n    "Species": "australis",\n    "Subspecies": "xenia",\n    "Sex": "M",\n    "Date": "20 Aug. 1929",\n    "Locality": "",\n    "PlaceName": "Ihosy",\n    "Region": "S.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n"RegistrationNumber": "1931.8.18.97",\n"Genus": "

In [ ]:
resized_image_outputs

['{\n    "RegistrationNumber": "1931.8.18.363",\n    "Genus": "Tyto",\n    "Species": "alba",\n    "Subspecies": "affinis (hypermetra)",\n    "Sex": "M",\n    "Date": "18 May 1930",\n    "Locality": "",\n    "PlaceName": "Maroantsetra",\n    "Region": "N.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n    "RegistrationNumber": "1931.8.18.1866",\n    "Genus": "Zoonavena",\n    "Species": "grandidieri",\n    "Subspecies": "",\n    "Sex": "F",\n    "Date": "23 July 1929",\n    "Locality": "30 km west",\n    "PlaceName": "Vondrozo",\n    "Region": "S.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n    "RegistrationNumber": "1931.8.18.833",\n    "Genus": "Ixeron",\n    "Species": "australis",\n    "Subspecies": "xenia",\n    "Sex": "M",\n    "Date": "20 Aug. 1929",\n    "Locality": "",\n    "PlaceName": "Ihosy",\n    "Region": "S.E. Madagascar",\n    "Expedition": "Archbold-Vernay Exped."\n}',
 '{\n    "RegistrationNumber": "1931.8.18.97",\n    "G

## Final loop

In [ ]:
# Run
all_new_responses = {}

errors = []

k = 10 # Looks at first k images. If interested in whole folder then remove [:k] from below.

for folder in tqdm.tqdm(os.listdir(folder_path)):
    filelist = os.listdir(folder_path+'/'+folder)
    for image in filelist:
      if image != '.DS_Store':
        image_path = folder_path+'/'+folder+'/'+image
        try:
            payload = get_payload(image_path,prompt,encode_function=0)
            #encode_function=0 for the original images , = 1 for resized images
            response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
            text = response.json()['choices'][0]['message']['content']
            all_new_responses[image] = text
        except:
            errors.append([folder,image])

100%|██████████| 97/97 [18:51<00:00, 11.66s/it]


In [ ]:
# Run
errors

[['Tringa hypoleucos', '1931.8.18.911.JPG'],
 ['Tringa hypoleucos', '1931.8.18.912.JPG'],
 ['Tringa hypoleucos', '1931.8.18.917.JPG'],
 ['Tringa hypoleucos', '1931.8.18.916.JPG'],
 ['Tringa hypoleucos', '1931.8.18.915.JPG'],
 ['Tringa hypoleucos', '1931.8.18.913.JPG'],
 ['Tringa hypoleucos', '1931.8.18.914.JPG'],
 ['Treron australis xenia', '1931.8.18.832.JPG'],
 ['Treron australis xenia', '1931.8.18.831.JPG'],
 ['Treron australis xenia', '1931.8.18.833.JPG'],
 ['Treron australis xenia', '1931.8.18.829.JPG'],
 ['Treron australis xenia', '1931.8.18.830.JPG'],
 ['Treron australis xenia', '1931.8.18.826.JPG'],
 ['Treron australis xenia', '1931.8.18.825.JPG'],
 ['Treron australis xenia', '1931.8.18.827.JPG'],
 ['Treron australis xenia', '1931.8.18.828.JPG'],
 ['Tyto alba affinis', '1931.8.18.351.JPG'],
 ['Tyto alba affinis', '1931.8.18.350.JPG'],
 ['Tyto alba affinis', '1931.8.18.352.JPG'],
 ['Tyto alba affinis', '1931.8.18.353.JPG'],
 ['Thalassornis leucomatus insularis', '1931.8.18.3712.

In [ ]:
# Run
outputs_formatted = []
formatting_errors = []
file_names=[]
for file in all_new_responses.keys():
    try:
        outputs_formatted.append(json.loads(all_new_responses[file]))
        file_names.append(file)
    except:
        try:

            result,_ = reformat_output(all_new_responses[file])
            outputs_formatted.append(json.loads(result))
            file_names.append(file)
        except:
            formatting_errors.append(file)


In [ ]:
# Run
formatting_errors

[]

In [ ]:
# Run
# Save into dataframe
outputs_df = pd.DataFrame(outputs_formatted)
outputs_df.insert(0, 'Filename',file_names )

# Save as CSV
outputs_df.to_csv('/content/drive/My Drive/Madagascar birds/llm_test_results.csv',index=False)